Groundwater | Flow Modeling Track

# Step 5: Model Calibration

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → 4-Build → **5-Calibrate** → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Steps 1-4, you defined objectives, developed a perceptual model, learned MODFLOW 6 fundamentals, and built a numerical model. Now we **calibrate** the model – adjusting parameters so simulated heads match observed measurements.

| **Core content** | ~50-60 minutes |
|:--|:--|
| **Optional: Non-uniqueness exploration** | +10-15 minutes |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Load and visualize** observation data for model calibration
2. **Calculate** calibration metrics (ME, MAE, RMSE, NRMSE, R²)
3. **Interpret** residual patterns to guide parameter adjustments
4. **Perform** manual calibration by adjusting hydraulic parameters
5. **Verify** calibration quality through metrics and water balance checks
6. **Understand** non-uniqueness (equifinality) in groundwater calibration

## Prerequisites

- **Completed [4_model_implementation.ipynb](4_model_implementation.ipynb)** – working MODFLOW 6 model
- Understanding of hydraulic conductivity and model parameters
- Basic statistics (mean, standard deviation, correlation)

---

## What is Calibration?

**Calibration** is the process of adjusting model parameters within physically reasonable bounds to minimize the difference between simulated and observed values.

In groundwater modeling, we typically calibrate against:
- **Hydraulic heads** at observation wells (primary target)
- **Groundwater fluxes** (e.g., baseflow to rivers, spring discharge)
- **Contaminant concentrations** (for transport models)

### The Calibration Workflow

```
1. Define targets    →  Select observations, define metrics
2. Run model         →  Execute with initial parameters
3. Compare outputs   →  Calculate residuals (sim - obs)
4. Adjust parameters →  Modify K, recharge, etc.
5. Iterate           →  Repeat until fit is acceptable
6. Evaluate          →  Check physical plausibility & water balance
```

> **Calibration vs. Validation**
>
> Never use all data for calibration! Reserve some observations for independent **validation** (Notebook 6). A model that fits calibration data but fails validation has been **overfit** and cannot be trusted for predictions.

---

## 1 - Setup

In [ ]:
# Setup - import required libraries
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Scientific computing
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# FloPy - MODFLOW 6
import flopy
from flopy.mf6 import MFSimulation

# Course utilities
sys.path.insert(0, '../../_SUPPORT/src')
sys.path.insert(0, '../../_SUPPORT/src/scripts/scripts_exercises')

from data_utils import download_named_file, get_default_data_folder
from model_io_utils import format_budget_summary
from calibration_utils import (
    calculate_calibration_metrics,
    format_metrics_table,
    load_awel_observations,
    generate_synthetic_observations,
    combine_observations,
    extract_heads_at_observations,
    plot_calibration_scatter,
    plot_residual_map,
    plot_observation_network,
    get_observation_summary,
)
from plot_utils import quick_model_plot

# Checkpoint utilities
from shared_functions import check_task_with_solution, create_multiple_choice

print(f"FloPy version: {flopy.__version__}")

In [ ]:
# Define paths
DATA_DIR = get_default_data_folder()
MODEL_WS = os.path.join(DATA_DIR, 'notebook4_model')
CALIB_WS = os.path.join(DATA_DIR, 'notebook5_calibration')

# Create calibration workspace
if not os.path.exists(CALIB_WS):
    os.makedirs(CALIB_WS)

print(f"Model source: {MODEL_WS}")
print(f"Calibration workspace: {CALIB_WS}")

---

## 2 - Load Model from Notebook 4

We load the MODFLOW 6 model created in Notebook 4. This model has:
- DISV (Voronoi) grid with river-aligned cells
- Initial parameter estimates (K, recharge, river conductance)
- Boundary conditions (CHD, WEL, RIV, RCH)

In [ ]:
# Load the MODFLOW 6 simulation from Notebook 4
try:
    sim = MFSimulation.load(
        sim_name='mfsim',
        sim_ws=MODEL_WS,
        verbosity_level=0
    )
    print("Simulation loaded successfully!")
except Exception as e:
    print(f"Error loading simulation: {e}")
    print("\nPlease ensure you have completed Notebook 4 first.")
    print(f"Expected model location: {MODEL_WS}")
    raise

In [ ]:
# Get the groundwater flow model
model_names = list(sim.model_names)
print(f"Models in simulation: {model_names}")

gwf_name = model_names[0]
gwf = sim.get_model(gwf_name)

print(f"\nModel: {gwf_name}")
print(f"Grid type: {type(gwf.modelgrid).__name__}")
print(f"Number of cells: {gwf.modelgrid.ncpl}")
print(f"Packages: {[pkg.name[0] for pkg in gwf.packagelist]}")

In [ ]:
# Get the model grid for later use
model_grid = gwf.modelgrid

# Get idomain (active cells)
disv = gwf.get_package('DISV')
idomain = disv.idomain.get_data()

n_active = np.sum(idomain > 0)
print(f"Active cells: {n_active}")

### 2.1 - Run Initial Model

First, let's run the model with its current (uncalibrated) parameters to establish a baseline.

In [ ]:
# Run the model
print("Running initial model...")
sim.write_simulation()
success, buff = sim.run_simulation(silent=True)

if success:
    print("Model run completed successfully!")
else:
    print("Model run failed!")
    print(buff)

In [ ]:
# Read initial heads
head_file = gwf.output.head()
initial_heads = head_file.get_data()

print(f"Head array shape: {initial_heads.shape}")
print(f"Head range: {np.nanmin(initial_heads):.2f} to {np.nanmax(initial_heads):.2f} m")

---

## 3 - Load Observation Data

We use groundwater level observations from two sources:

1. **Real observations** from the AWEL (Amt für Wasser, Energie und Luft) monitoring network
2. **Synthetic observations** added for teaching purposes

> **About the Observation Data**
>
> This notebook uses two types of observation data:
>
> 1. **Real observations** (marked `is_synthetic=False`): Groundwater levels from the Canton Zurich monitoring network (AWEL). These are actual measurements from the Limmat Valley.
>
> 2. **Synthetic observations** (marked `is_synthetic=True`): Artificial data points we created to ensure adequate spatial coverage for demonstrating calibration methods. These were generated by:
>    - Running the model with "true" parameter values
>    - Extracting heads at strategically placed locations
>    - Adding realistic measurement noise (±0.3 m)
>
> In professional practice, you would only use real observations. The synthetic points here serve an educational purpose: they allow us to demonstrate calibration concepts even when real data coverage is limited.

### 3.1 - Load Model Boundary

We need the model boundary to filter observations to our domain.

In [ ]:
# Load model boundary
boundary_file = download_named_file('model_boundary', data_type='gis')
boundary_gdf = gpd.read_file(boundary_file)

print(f"Boundary CRS: {boundary_gdf.crs}")
print(f"Boundary area: {boundary_gdf.geometry.area.iloc[0] / 1e6:.2f} km²")

### 3.2 - Load Real AWEL Observations

The AWEL monitoring network provides groundwater level measurements from observation wells across the Limmat Valley.

In [ ]:
# Load AWEL observation data
try:
    real_obs = load_awel_observations(
        data_dir=DATA_DIR,
        model_boundary=boundary_gdf,
        use_mean=True  # Use mean head for steady-state calibration
    )
    print(f"Loaded {len(real_obs)} real observation wells")
    print(real_obs[['obs_id', 'head_m', 'is_synthetic']].to_string())
except FileNotFoundError as e:
    print(f"AWEL data not found: {e}")
    print("Creating empty real observations dataframe...")
    real_obs = gpd.GeoDataFrame(
        columns=['obs_id', 'x', 'y', 'head_m', 'error_m', 'is_synthetic', 'source', 'geometry'],
        crs="EPSG:2056"
    )

### 3.3 - Generate Synthetic Observations

To ensure adequate spatial coverage for demonstrating calibration, we add synthetic observation points. These are generated from a reference model run with added measurement noise.

In [ ]:
# Generate synthetic observations
# We use a "true" model run as reference (the initial model serves as our reference)
# In practice, we would use known "true" parameters, but for teaching purposes
# we treat the initial model as providing the "true" heads at synthetic locations

# Get river cell indices to exclude - observations on river cells are not meaningful
# because their heads are controlled by the RIV boundary condition
riv_package = gwf.get_package('RIV')
if riv_package is not None:
    riv_spd = riv_package.stress_period_data.get_data()
    # Extract cell IDs from RIV stress period data
    river_cells = set()
    for per, data in riv_spd.items():
        if data is not None:
            # Cell ID is typically in 'cellid' column for DISV grids
            for rec in data:
                cellid = rec['cellid']
                if isinstance(cellid, tuple):
                    river_cells.add(cellid[-1])  # Get cell number from (layer, cell) tuple
                else:
                    river_cells.add(cellid)
    river_cells = np.array(list(river_cells))
    print(f"Found {len(river_cells)} river cells to exclude")
else:
    river_cells = None
    print("No RIV package found")

# Load river geometries
try:
    rivers_file = download_named_file('rivers', data_type='gis')
    rivers_gdf = gpd.read_file(rivers_file)
    print(f"Loaded river geometries for buffering")
except:
    rivers_gdf = None
    print("Could not load river geometries")

# Generate synthetic observations away from rivers
synth_obs = generate_synthetic_observations(
    model_grid=model_grid,
    true_heads=initial_heads,
    idomain=idomain,
    n_points=15,              # Number of synthetic observation points
    noise_std=0.3,            # Measurement noise (m)
    seed=42,                  # For reproducibility
    min_distance_m=300.0,     # Minimum spacing between points
    avoid_boundaries_m=150.0, # Stay away from domain edges
    exclude_cells=river_cells, # Exclude river cells
    river_buffer_m=75.0,      # Stay 75m away from river polygons
    river_gdf=rivers_gdf      # River geometries
)

print(f"\nGenerated {len(synth_obs)} synthetic observation points")
print(f"Noise added: ±0.3 m (1σ)")

### 3.4 - Combine Observations

In [ ]:
# Combine real and synthetic observations
obs_gdf = combine_observations(
    real_obs=real_obs,
    synthetic_obs=synth_obs,
    drop_true_heads=True  # Hide "answers" for calibration exercise
)

# Print summary
print(get_observation_summary(obs_gdf))

In [ ]:
# Display observation data
obs_gdf[['obs_id', 'head_m', 'error_m', 'is_synthetic', 'source']]

### ✅ Checkpoint 1: Observation Count

In [ ]:
# Display observation count for reference
n_observations = len(obs_gdf)
print(f"Total observations: {n_observations}")
print("\nEnter this value in the checkpoint below:")

# Interactive checkpoint
check_task_with_solution('task05_checkpoint_1')

### 3.5 - Visualize Observation Network

In [ ]:
# Load rivers for context
try:
    rivers_file = download_named_file('rivers', data_type='gis')
    rivers_gdf = gpd.read_file(rivers_file)
except:
    rivers_gdf = None

# Plot observation network
fig = plot_observation_network(
    obs_gdf=obs_gdf,
    boundary_gdf=boundary_gdf,
    river_gdf=rivers_gdf,
    title="Observation Network: Real (AWEL) and Synthetic Points",
    figsize=(12, 10)
)
plt.show()

---

## 4 - Calibration Metrics

Before we start calibrating, we need to define how we measure the quality of fit between simulated and observed values.

### Common Calibration Metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **ME** (Mean Error) | $\frac{1}{n}\sum(h_{sim} - h_{obs})$ | Bias - positive means sim > obs |
| **MAE** (Mean Absolute Error) | $\frac{1}{n}\sum|h_{sim} - h_{obs}|$ | Average error magnitude |
| **RMSE** (Root Mean Square Error) | $\sqrt{\frac{1}{n}\sum(h_{sim} - h_{obs})^2}$ | Sensitive to large errors |
| **NRMSE** (Normalized RMSE) | $\frac{RMSE}{h_{max} - h_{min}} \times 100$ | RMSE as % of range |
| **R²** (Coefficient of Determination) | $1 - \frac{SS_{res}}{SS_{tot}}$ | Goodness of fit (0-1) |

### Calibration Targets

- **NRMSE < 10%**: Generally acceptable for regional models
- **R² > 0.9**: Good correlation between simulated and observed
- **ME ≈ 0**: No systematic bias

---

## 5 - Initial Model Assessment

Let's evaluate how well the uncalibrated model matches observations.

In [ ]:
# Extract simulated heads at observation locations
simulated_initial = extract_heads_at_observations(
    head_array=initial_heads,
    obs_gdf=obs_gdf,
    model_grid=model_grid
)

# Get observed heads
observed = obs_gdf['head_m'].values

# Check for valid values
valid = ~np.isnan(simulated_initial)
print(f"Valid observations: {valid.sum()} of {len(valid)}")

In [ ]:
# Calculate initial metrics
initial_metrics = calculate_calibration_metrics(
    observed[valid], 
    simulated_initial[valid]
)

print("INITIAL MODEL ASSESSMENT")
print(format_metrics_table(initial_metrics))

### ✅ Checkpoint 2: Initial RMSE

In [ ]:
# Display initial RMSE for reference
initial_rmse = initial_metrics['RMSE']
print(f"Initial RMSE: {initial_rmse:.2f} m")
print("\nEnter this value in the checkpoint below:")

# Interactive checkpoint
check_task_with_solution('task05_checkpoint_2')

### 5.1 - Scatter Plot: Observed vs Simulated

In [ ]:
# Create scatter plot
fig = plot_calibration_scatter(
    observed=observed[valid],
    simulated=simulated_initial[valid],
    obs_gdf=obs_gdf[valid],
    title="Initial Model: Observed vs Simulated Heads",
    show_metrics=True
)
plt.show()

### 5.2 - Residual Map

The spatial distribution of residuals tells us which areas need parameter adjustments.

In [ ]:
# Calculate residuals
initial_residuals = simulated_initial - observed

# Plot residual map
fig = plot_residual_map(
    obs_gdf=obs_gdf,
    residuals=initial_residuals,
    boundary_gdf=boundary_gdf,
    title="Initial Model: Spatial Residuals (Simulated - Observed)",
    figsize=(12, 10)
)
plt.show()

---

## 6 - Understanding Residual Patterns

Residual patterns guide our calibration adjustments:

| Residual Pattern | Likely Cause | Adjustment |
|-----------------|--------------|------------|
| **Positive everywhere** (sim > obs) | K too low or recharge too high | Increase K or decrease recharge |
| **Negative everywhere** (sim < obs) | K too high or recharge too low | Decrease K or increase recharge |
| **Positive in one zone** | Zone K too low | Increase K in that zone |
| **Pattern near rivers** | River conductance wrong | Adjust river bed K |
| **Random scatter** | Measurement errors, heterogeneity | May be at achievable limit |

### Key Insight: The K-Recharge Trade-off

For steady-state conditions, head is approximately:

$$h \propto \frac{\text{Recharge}}{K}$$

This means:
- **Increase K** → heads decrease (water drains faster)
- **Increase Recharge** → heads increase (more water input)
- **K and Recharge are correlated** in calibration (non-uniqueness!)

### ✅ Checkpoint 5: Residual Interpretation

In [ ]:
# Conceptual question about residual interpretation
create_multiple_choice('task05_checkpoint_5')

---

## 7 - Manual Calibration

Now we'll adjust parameters to improve the fit. We use **multipliers** to scale the original parameter values.

### Calibration Strategy

1. Start with global K multiplier
2. Adjust recharge if needed
3. Fine-tune river conductance
4. Iterate until RMSE is acceptable

### 7.1 - Define Calibration Trial Function

In [ ]:
def evaluate_trial(k_mult, rch_mult=1.0, riv_mult=1.0, verbose=True):
    """
    Run a calibration trial with given multipliers and return metrics.
    
    Parameters
    ----------
    k_mult : float
        Multiplier for hydraulic conductivity (K)
    rch_mult : float
        Multiplier for recharge
    riv_mult : float
        Multiplier for river conductance
    verbose : bool
        Print results
    
    Returns
    -------
    dict : Calibration metrics
    """
    # Reload simulation to get fresh parameters
    sim_trial = MFSimulation.load(sim_ws=MODEL_WS, verbosity_level=0)
    gwf_trial = sim_trial.get_model(gwf_name)
    
    # Modify K
    if k_mult != 1.0:
        npf = gwf_trial.get_package('NPF')
        k_orig = npf.k.get_data()
        npf.k.set_data(k_orig * k_mult)
    
    # Modify recharge
    if rch_mult != 1.0:
        rch = gwf_trial.get_package('RCHA')
        if rch is not None:
            rch_data = rch.recharge.get_data()
            # Handle dict format (stress period data)
            if isinstance(rch_data, dict):
                new_rch_data = {}
                for per, arr in rch_data.items():
                    if arr is not None:
                        new_rch_data[per] = arr * rch_mult
                    else:
                        new_rch_data[per] = arr
                rch.recharge.set_data(new_rch_data)
            else:
                # Simple array format
                rch.recharge.set_data(rch_data * rch_mult)
    
    # Modify river conductance
    if riv_mult != 1.0:
        riv = gwf_trial.get_package('RIV')
        if riv is not None:
            spd = riv.stress_period_data.get_data()
            for per, data in spd.items():
                if data is not None:
                    data['cond'] = data['cond'] * riv_mult
            riv.stress_period_data.set_data(spd)
    
    # Run model
    sim_trial.write_simulation()
    success, _ = sim_trial.run_simulation(silent=True)
    
    if not success:
        if verbose:
            print(f"Trial failed: K={k_mult:.2f}, RCH={rch_mult:.2f}, RIV={riv_mult:.2f}")
        return None
    
    # Get heads and calculate metrics
    heads = gwf_trial.output.head().get_data()
    simulated = extract_heads_at_observations(heads, obs_gdf, model_grid)
    
    valid_mask = ~np.isnan(simulated)
    metrics = calculate_calibration_metrics(
        observed[valid_mask], 
        simulated[valid_mask]
    )
    metrics['k_mult'] = k_mult
    metrics['rch_mult'] = rch_mult
    metrics['riv_mult'] = riv_mult
    metrics['heads'] = heads
    metrics['simulated'] = simulated
    
    if verbose:
        print(f"K={k_mult:.2f}, RCH={rch_mult:.2f}, RIV={riv_mult:.2f} -> "
              f"RMSE={metrics['RMSE']:.3f} m, R²={metrics['R2']:.3f}")
    
    return metrics

### 7.2 - Systematic Calibration Trials

Let's run a series of trials varying the K multiplier.

In [ ]:
# Define K multipliers to test
k_multipliers = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]

# Run trials
print("Running calibration trials...")
print("=" * 60)

trial_results = []
for k_mult in k_multipliers:
    result = evaluate_trial(k_mult, verbose=True)
    if result is not None:
        trial_results.append(result)

print("=" * 60)

In [ ]:
# Create results dataframe
results_df = pd.DataFrame([
    {'k_mult': r['k_mult'], 'RMSE': r['RMSE'], 'R2': r['R2'], 'ME': r['ME'], 'NRMSE': r['NRMSE']}
    for r in trial_results
])

print("\nCalibration Trial Results:")
print(results_df.to_string(index=False))

In [ ]:
# Plot RMSE vs K multiplier
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(results_df['k_mult'], results_df['RMSE'], 'bo-', markersize=10, linewidth=2)
ax.axhline(y=2.0, color='r', linestyle='--', label='Target RMSE (2 m)')

# Mark minimum
min_idx = results_df['RMSE'].idxmin()
ax.scatter(results_df.loc[min_idx, 'k_mult'], results_df.loc[min_idx, 'RMSE'],
           color='green', s=200, zorder=5, label=f"Best: K×{results_df.loc[min_idx, 'k_mult']:.2f}")

ax.set_xlabel('K Multiplier', fontsize=12)
ax.set_ylabel('RMSE (m)', fontsize=12)
ax.set_title('Calibration: RMSE vs Hydraulic Conductivity Multiplier', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7.3 - Select Best Parameters

In [ ]:
# Find best trial
best_idx = results_df['RMSE'].idxmin()
best_k_mult = results_df.loc[best_idx, 'k_mult']
best_result = trial_results[best_idx]

print(f"Best K multiplier: {best_k_mult}")
print(f"Best RMSE: {best_result['RMSE']:.3f} m")
print(f"Best R²: {best_result['R2']:.3f}")

### 7.4 - ✏️ Exercise: Fine-tune Calibration

Try different combinations of K and recharge multipliers to improve the fit further.

In [ ]:
# Try your own parameter combinations
# Uncomment and modify the values below

# my_result = evaluate_trial(
#     k_mult=1.2,      # Try values around the best K multiplier
#     rch_mult=0.9,    # Try adjusting recharge
#     riv_mult=1.0     # Try adjusting river conductance
# )

---

## 8 - Calibration Results

Let's examine the calibrated model in detail.

In [ ]:
# Get calibrated heads and residuals
calibrated_heads = best_result['heads']
calibrated_simulated = best_result['simulated']
calibrated_residuals = calibrated_simulated - observed

print("CALIBRATED MODEL RESULTS")
print("=" * 50)
print(f"K multiplier: {best_result['k_mult']}")
print(f"Effective K: {20 * best_result['k_mult']:.1f} m/day (from 20 m/day base)")
print()
print(format_metrics_table(best_result))

### ✅ Checkpoint 3: Calibrated RMSE

In [ ]:
# Display calibrated RMSE for reference
calibrated_rmse = best_result['RMSE']
print(f"Calibrated RMSE: {calibrated_rmse:.3f} m")
print("\nEnter this value in the checkpoint below:")

# Interactive checkpoint
check_task_with_solution('task05_checkpoint_3')

### 8.1 - Compare Initial vs Calibrated

In [ ]:
# Side-by-side scatter plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Initial model
ax = axes[0]
ax.scatter(observed[valid], simulated_initial[valid], c='#2E86AB', s=60, alpha=0.7)
ax.plot([observed.min(), observed.max()], [observed.min(), observed.max()], 'k--')
ax.set_xlabel('Observed Head (m)')
ax.set_ylabel('Simulated Head (m)')
ax.set_title(f'Initial Model\nRMSE = {initial_metrics["RMSE"]:.2f} m, R² = {initial_metrics["R2"]:.3f}')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Calibrated model
ax = axes[1]
ax.scatter(observed[valid], calibrated_simulated[valid], c='#27AE60', s=60, alpha=0.7)
ax.plot([observed.min(), observed.max()], [observed.min(), observed.max()], 'k--')
ax.set_xlabel('Observed Head (m)')
ax.set_ylabel('Simulated Head (m)')
ax.set_title(f'Calibrated Model (K×{best_k_mult})\nRMSE = {best_result["RMSE"]:.2f} m, R² = {best_result["R2"]:.3f}')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 8.2 - Calibrated Residual Map

In [ ]:
# Plot calibrated residuals
fig = plot_residual_map(
    obs_gdf=obs_gdf,
    residuals=calibrated_residuals,
    boundary_gdf=boundary_gdf,
    title=f"Calibrated Model: Residuals (K×{best_k_mult})",
    figsize=(12, 10)
)
plt.show()

---

## 9 - Water Balance Verification

A calibrated model must maintain water balance. Let's check the budget.

In [ ]:
# Run the calibrated model and get budget
sim_calib = MFSimulation.load(sim_ws=MODEL_WS, verbosity_level=0)
gwf_calib = sim_calib.get_model(gwf_name)

# Apply calibrated K
npf = gwf_calib.get_package('NPF')
k_orig = npf.k.get_data()
npf.k.set_data(k_orig * best_k_mult)

# Run and get budget
sim_calib.write_simulation()
success, _ = sim_calib.run_simulation(silent=True)

# Initialize balance_error
balance_error = None

if success:
    # Use the format_budget_summary utility function
    budget_df = format_budget_summary(gwf_calib, sim_calib)
    
    print("\nWater Balance Summary")
    print("=" * 60)
    print(budget_df.to_string())
    
    # Extract balance error from the DataFrame
    if 'PERCENT ERROR' in budget_df.index:
        balance_error = float(budget_df.loc['PERCENT ERROR', 'Net (m3/d)'])
    else:
        # Fallback calculation
        total_in = budget_df.loc['TOTAL', 'Inflow (m3/d)']
        total_out = budget_df.loc['TOTAL', 'Outflow (m3/d)']
        balance_error = abs(total_in - total_out) / ((total_in + total_out) / 2) * 100
    
    print(f"\nWater Balance Error: {balance_error:.4f}%")
else:
    print("Model run failed - cannot compute water balance")

### ✅ Checkpoint 4: Water Balance Error

In [ ]:
# Display water balance error for reference
if balance_error is not None:
    print(f"Water balance error: {balance_error:.4f}%")
    print("\nEnter this value in the checkpoint below:")
    
    # Interactive checkpoint
    check_task_with_solution('task05_checkpoint_4')
else:
    print("Water balance could not be computed - skipping checkpoint")

---

## 10 - Non-Uniqueness (Equifinality)

**Non-uniqueness** means that different parameter combinations can produce similar calibration fits. This is a fundamental challenge in groundwater modeling.

### The K-Recharge Trade-off

For steady-state flow, if you increase K by a factor and increase recharge by the same factor, the heads remain approximately the same:

$$h \approx f\left(\frac{R}{K}\right)$$

This means calibration alone cannot uniquely determine both K and recharge.

In [ ]:
# Demonstrate non-uniqueness
# Try different K-recharge combinations that maintain similar ratio

print("Demonstrating Non-Uniqueness (K-Recharge Trade-off)")
print("=" * 60)

# Original calibrated result
print(f"\nBase case: K×{best_k_mult:.2f}, RCH×1.00 -> RMSE = {best_result['RMSE']:.3f} m")

# Try compensating combinations
combinations = [
    (best_k_mult * 0.8, 0.8),   # Both reduced
    (best_k_mult * 1.2, 1.2),   # Both increased
]

for k_mult, rch_mult in combinations:
    result = evaluate_trial(k_mult, rch_mult, verbose=False)
    if result:
        print(f"Alternative: K×{k_mult:.2f}, RCH×{rch_mult:.2f} -> RMSE = {result['RMSE']:.3f} m")

print("\n-> Similar RMSE with different parameters = NON-UNIQUENESS")
print("   This is why we need independent constraints (pumping tests, tracer tests)!")

---

## 11 - Summary and Next Steps

### What We Accomplished

1. **Loaded** observation data (real AWEL + synthetic points)
2. **Assessed** initial model fit using calibration metrics
3. **Interpreted** residual patterns to guide adjustments
4. **Calibrated** the model by adjusting K multipliers
5. **Verified** water balance closure
6. **Explored** non-uniqueness (equifinality)

### Calibration Results Summary

In [ ]:
# Final summary
print("CALIBRATION SUMMARY")
print("=" * 50)
print(f"\nInitial Model:")
print(f"  RMSE:  {initial_metrics['RMSE']:.3f} m")
print(f"  R²:    {initial_metrics['R2']:.3f}")
print(f"  NRMSE: {initial_metrics['NRMSE']:.1f}%")

print(f"\nCalibrated Model (K×{best_k_mult}):")
print(f"  RMSE:  {best_result['RMSE']:.3f} m")
print(f"  R²:    {best_result['R2']:.3f}")
print(f"  NRMSE: {best_result['NRMSE']:.1f}%")

print(f"\nImprovement:")
print(f"  RMSE reduced by {(1 - best_result['RMSE']/initial_metrics['RMSE'])*100:.1f}%")
if balance_error is not None:
    print(f"  Water balance error: {balance_error:.4f}%")

if best_result['NRMSE'] < 10:
    print("\n✅ Calibration target achieved (NRMSE < 10%)")
else:
    print("\n⚠️ Calibration could be improved (NRMSE > 10%)")

### Key Takeaways

1. **Calibration is iterative** - expect multiple rounds of adjustment
2. **Physical plausibility matters** - don't chase fit at expense of realism
3. **Non-uniqueness is real** - calibration alone doesn't uniquely determine parameters
4. **Residual patterns guide adjustments** - use spatial information
5. **Always verify water balance** - a good fit means nothing if mass isn't conserved

### Next Steps

- **Notebook 6 (Validation)**: Test the calibrated model against independent data
- **Notebook 7 (Sensitivity)**: Understand which parameters matter most
- **Notebook 8 (Application)**: Use the model for predictions

---

**End of Notebook 5**